# Oxta — Smoke Test 100% Synthetic

**Objetivo único:** validar que a arquitetura completa do Oxta (Mamba2 + Attention + MoE + BF16) treina e aprende. Sem datasets externos, sem HF_TOKEN, sem auth de nada além do clone do repo.

**Como:** geramos 2000 frases sintéticas em PT-BR com vocabulário fiscal usando templates (CNPJ, valores, datas, NF, etc.) — em Python puro, ~3 segundos. Treinamos 500 steps. Se loss desce, o stack tá vivo.

**Tempo total:** ~20-30 min em T4 (build cached) ou ~35-45 min (build cold).

## Stack que vamos exercitar

| Componente | Estado |
|-----------|--------|
| Mamba2 SSD | ✅ ON (base layers) |
| Attention GQA | ✅ ON (3 layers) |
| MoE-8 top-2 | ✅ ON (2 layers) |
| BF16 Tensor Cores | ✅ ON |
| Gradient checkpointing | ✅ ON |
| CUDA + AVX2 | ✅ ON |
| QAT 1.58-bit | ⚠️ OFF (binding não expõe — treino em BF16, ok pra smoke) |

**Critério de sucesso:** loss inicial ~ln(vocab) ≈ 8-10, cai pra <7 em 500 steps.

## 1. Setup mínimo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_OWNER = 'vitorGgC569'
REPO_NAME = 'reimagined'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')

# GITHUB_TOKEN do Colab Secrets (você já configurou)
clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        clone_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('[auth] GITHUB_TOKEN OK')
except Exception:
    pass

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    clone_url, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'set-url',
                    'origin', clone_url], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'colab'))
os.environ['NSOS_PARAM_SCALE'] = '80m'
os.environ['NSOS_MIXED_PRECISION'] = 'bf16'

import colab_bootstrap
env = colab_bootstrap.setup_environment(verbose=True)
BUILD_DIR = Path(env['build_dir'])
print(f'\nBuild OK on {env["gpu_name"]}, BF16: {os.environ["NSOS_MIXED_PRECISION"]}')

In [ ]:
sys.path.insert(0, str(BUILD_DIR))
if 'nsos_ext' in sys.modules:
    del sys.modules['nsos_ext']
import nsos_ext
print(f'nsos_ext: {nsos_ext.__file__}')
print(f'fast_gpu_supported: {nsos_ext.fast_gpu_supported()}')

## 2. Gerar corpus sintético — Python puro, zero downloads

2000 frases construídas a partir de templates com vocabulário fiscal brasileiro real. Variedade suficiente pra modelo ter o que aprender.

In [ ]:
import random

random.seed(1337)

NOMES = [
    'João Silva', 'Maria Santos', 'Carlos Oliveira', 'Ana Costa', 'Pedro Souza',
    'Beatriz Lima', 'Rafael Alves', 'Camila Rodrigues', 'Lucas Pereira',
    'Fernanda Martins', 'Bruno Ferreira', 'Patrícia Gomes', 'André Carvalho',
    'Juliana Ribeiro', 'Marcos Almeida', 'Renata Barbosa', 'Eduardo Pinto',
]
EMPRESAS = [
    'Padaria do Bairro Ltda', 'Comércio Central S/A', 'Tech Solutions ME',
    'Distribuidora ABC', 'Construtora Horizonte', 'Mercado Vila Nova',
    'Auto Peças do Rio', 'Farmácia Saúde Plus', 'Restaurante Sabor Caseiro',
    'Escola Aprender Mais', 'Clínica Bem Estar', 'Posto de Combustível 24h',
    'Loja de Roupas Estilo', 'Papelaria Universitária',
]
CIDADES = [
    'São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Salvador', 'Brasília',
    'Curitiba', 'Fortaleza', 'Manaus', 'Porto Alegre', 'Recife', 'Goiânia',
]
TIPOS_RENDIMENTO = [
    'salário', 'aluguel', 'pensão alimentícia', 'pró-labore', 'dividendos',
    'aposentadoria', 'rendimento de aplicação financeira', 'férias',
]
TIPOS_DESPESA = [
    'plano de saúde', 'educação', 'previdência privada', 'dependentes',
    'pensão alimentícia paga', 'livro-caixa', 'instrução',
]

def gen_cnpj():
    return f'{random.randint(10,99)}.{random.randint(100,999):03d}.{random.randint(100,999):03d}/0001-{random.randint(10,99):02d}'

def gen_cpf():
    return f'{random.randint(100,999)}.{random.randint(100,999):03d}.{random.randint(100,999):03d}-{random.randint(10,99):02d}'

def gen_valor(low=100, high=200000):
    v = random.randint(low, high)
    # Formata BR: ponto milhar, vírgula decimal
    return 'R$ ' + f'{v:,}'.replace(',', '.') + ',00'

def gen_data():
    return f'{random.randint(1,28):02d}/{random.randint(1,12):02d}/{random.randint(2022,2025)}'

def gen_nf():
    return f'{random.randint(1, 99999):05d}'

TEMPLATES = [
    'O contribuinte {nome}, CPF {cpf}, declarou rendimentos de {valor} no exercício 2024, referente a {tipo_rend} recebido durante o ano-calendário de 2023.',
    'A nota fiscal eletrônica número {nf}, emitida pela empresa {empresa}, CNPJ {cnpj}, em {data}, registra valor total de {valor} com destaque do ICMS no valor de {valor2}.',
    'A pessoa jurídica {empresa}, inscrita no CNPJ {cnpj}, sediada em {cidade}, apresentou faturamento bruto de {valor} no terceiro trimestre, sujeito ao regime de Lucro Presumido.',
    'Para o ano-base 2023, {nome} (CPF {cpf}) deduziu despesas com {tipo_desp} no valor total de {valor}, conforme comprovantes anexados à declaração.',
    'O imposto de renda apurado para o exercício 2024 do contribuinte {nome} totaliza {valor}, com restituição prevista no segundo lote pago em {data}.',
    'A empresa {empresa} (CNPJ {cnpj}) emitiu boleto bancário no valor de {valor} com vencimento em {data}, referente à prestação de serviços contábeis.',
    'O acórdão do CARF nº {nf}/2023 julgou improcedente o auto de infração lavrado contra {empresa}, no valor de {valor}, por entender configurada a decadência do crédito tributário.',
    'A guia DARF do código 0190 com vencimento em {data} no valor de {valor}, emitida em nome de {empresa}, refere-se ao IRPJ apurado pelo Lucro Real.',
    'O SPED Fiscal entregue pela empresa {empresa} para o período de apuração de {data} contém {nf} registros de notas fiscais com valor total movimentado de {valor}.',
    'A retenção na fonte do imposto de renda incidente sobre o pagamento de {valor} efetuado pela {empresa} a {nome} foi recolhida em {data} mediante DARF código 0561.',
    'A declaração de ajuste anual entregue por {nome} apresenta saldo de imposto a pagar de {valor}, parcelável em até oito quotas mensais com vencimento da primeira em {data}.',
    'O regime do Simples Nacional aplicável à {empresa}, com faturamento anual de {valor}, determina alíquota efetiva conforme tabela do anexo correspondente à atividade exercida.',
    'Para fins de comprovação de despesa médica, {nome} apresentou recibo emitido em {data} pela {empresa} no valor de {valor} referente a {tipo_desp}.',
    'A escrituração contábil digital (ECD) transmitida pela empresa {empresa} contempla o balanço patrimonial encerrado em 31/12/2023 com ativo total de {valor}.',
]

def gen_sentence():
    t = random.choice(TEMPLATES)
    return t.format(
        nome=random.choice(NOMES),
        empresa=random.choice(EMPRESAS),
        cidade=random.choice(CIDADES),
        cnpj=gen_cnpj(),
        cpf=gen_cpf(),
        valor=gen_valor(),
        valor2=gen_valor(50, 20000),
        data=gen_data(),
        nf=gen_nf(),
        tipo_rend=random.choice(TIPOS_RENDIMENTO),
        tipo_desp=random.choice(TIPOS_DESPESA),
    )

# Gera 2000 frases — agrupa 3 por doc pra ter contexto mais longo
N_SENTENCES = 2000
SENTENCES_PER_DOC = 3

sentences = [gen_sentence() for _ in range(N_SENTENCES)]
docs = []
for i in range(0, len(sentences), SENTENCES_PER_DOC):
    docs.append(' '.join(sentences[i:i + SENTENCES_PER_DOC]))

print(f'Geradas {len(sentences)} frases em {len(docs)} docs')
print(f'Tamanho médio por doc: {sum(len(d) for d in docs) / len(docs):.0f} chars')
print(f'Total de chars: {sum(len(d) for d in docs):,}')
print('\n--- exemplo doc 1 ---')
print(docs[0])
print('\n--- exemplo doc 2 ---')
print(docs[1])

## 3. ModelConfig — stack completo, ~123M params

In [ ]:
config = nsos_ext.ModelConfig()
config.num_layers          = 12
config.d_model             = 768
config.vocab_size          = 16384
config.max_context_tokens  = 1024

# Attention GQA: 3 de 12 layers
config.n_heads             = 12
config.n_kv_heads          = 3
config.sliding_window      = 2048
config.attention_period    = 4
config.attention_slot      = 3
config.use_exact_attention_training = True
config.use_flash_attn      = False

# MoE-8 top-2: 2 de 12 layers
config.use_moe             = True
config.num_experts         = 8
config.num_experts_per_token = 2
config.moe_period          = 6
config.moe_slot            = 5

# Research/inference-only: OFF
config.use_ttt             = False
config.mcts_simulations    = 0

# Memória + training
config.use_gradient_checkpointing = True
config.dropout             = 0.0
config.use_cuda            = True
config.default_batch_size  = 1

L, d, V = config.num_layers, config.d_model, config.vocab_size
n_attn = L // config.attention_period
n_moe = L // config.moe_period
n_mamba = L - n_attn - n_moe
print(f'Layers: Mamba={n_mamba}, Attn={n_attn}, MoE={n_moe} (total {L})')
print(f'd_model={d}, vocab={V}, max_ctx={config.max_context_tokens}')
print(f'MoE: {config.num_experts} experts, top-{config.num_experts_per_token}')

## 4. Engine fresh init

In [ ]:
engine = nsos_ext.InferenceEngine()
print('Inicializando engine fresh...')
ok = engine.load_model('', config)
print(f'load_model: {"OK" if ok else "FALHOU"}')
if not ok:
    raise RuntimeError('Engine init falhou. Verifique ModelConfig acima.')
print(f'Memory: {engine.get_memory_usage()}')

## 5. Geração ANTES (baseline — texto aleatório esperado)

In [ ]:
BASELINE_PROMPTS = [
    'O contribuinte',
    'A nota fiscal eletrônica',
    'O imposto de renda',
]
MAX_TOKENS = 50
TEMP = 0.8

baseline = {}
for p in BASELINE_PROMPTS:
    try:
        baseline[p] = engine.generate(p, MAX_TOKENS, TEMP)
    except Exception as e:
        baseline[p] = f'(erro: {type(e).__name__}: {e})'
    print(f'[{p}]')
    print(f'  -> {baseline[p][:200]}\n')

## 6. Treino 500 steps

In [ ]:
import time
import math
import matplotlib.pyplot as plt

MAX_STEPS = int(os.environ.get('NSOS_SMOKE_STEPS', '500'))
LOG_EVERY = 25

random.seed(1337)
random.shuffle(docs)

losses = []
step_times = []
t_start = time.time()

print(f'Treinando {MAX_STEPS} steps em {len(docs)} docs sintéticos...\n')

for step in range(1, MAX_STEPS + 1):
    t0 = time.time()
    doc = docs[(step - 1) % len(docs)]
    try:
        result = engine.train_text(doc)
        loss = float(result) if isinstance(result, (int, float)) else float(result.get('loss', 'nan'))
    except Exception as e:
        print(f'[step {step}] train_text falhou: {type(e).__name__}: {e}')
        continue
    step_times.append(time.time() - t0)
    losses.append(loss)

    if step % LOG_EVERY == 0 or step == 1:
        recent = losses[-LOG_EVERY:]
        avg_loss = sum(recent) / len(recent)
        avg_step = sum(step_times[-LOG_EVERY:]) / min(LOG_EVERY, len(step_times))
        eta_min = (MAX_STEPS - step) * avg_step / 60
        print(f'[step {step:>4d}/{MAX_STEPS}] loss={avg_loss:.3f}  step={avg_step:.2f}s  eta={eta_min:.1f}min')

total_min = (time.time() - t_start) / 60
print(f'\nFIM: {len(losses)} steps em {total_min:.1f} min')

## 7. Loss plot + verdict

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(losses, alpha=0.4, label='step loss')
window = max(1, len(losses) // 25)
if len(losses) > window:
    smoothed = [sum(losses[max(0, i-window):i+1]) / min(i+1, window) for i in range(len(losses))]
    plt.plot(smoothed, color='red', linewidth=2, label=f'smoothed (w={window})')
plt.xlabel('step')
plt.ylabel('loss')
plt.title(f'Oxta Stack Smoke — synthetic corpus, full stack ({len(losses)} steps)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/oxta_synthetic_smoke_loss.png', dpi=110)
plt.show()

# Verdict
if not losses:
    print('❌ FALHA: zero steps completaram')
else:
    loss_init = sum(losses[:10]) / min(10, len(losses))
    loss_final = sum(losses[-25:]) / min(25, len(losses))
    drop = loss_init - loss_final
    drop_pct = (drop / loss_init * 100) if loss_init > 0 else 0
    nan_count = sum(1 for l in losses if math.isnan(l) or math.isinf(l))
    expected_initial = math.log(config.vocab_size)

    print(f'\n{"="*70}')
    if nan_count > len(losses) * 0.05:
        print(f'❌ FALHA: {nan_count} losses NaN/Inf (>5% do treino)')
    elif loss_init > expected_initial * 2 or loss_init < 0.1:
        print(f'⚠️ ANORMAL: loss inicial {loss_init:.2f} fora do range (~{expected_initial:.1f})')
    elif drop_pct < 15:
        print(f'⚠️ INSUFICIENTE: loss caiu {drop_pct:.1f}% ({loss_init:.2f} → {loss_final:.2f})')
    else:
        print(f'✅ PASSOU: loss caiu {drop_pct:.1f}% ({loss_init:.2f} → {loss_final:.2f})')
        print(f'   Stack funcional. Modelo aprendeu padrões fiscais sintéticos.')
    print(f'{"="*70}')
    print(f'  loss inicial:   {loss_init:.3f}')
    print(f'  loss final:     {loss_final:.3f}')
    print(f'  loss min:       {min(losses):.3f}')
    print(f'  loss max:       {max(losses):.3f}')
    print(f'  NaN/Inf:        {nan_count}')
    print(f'  vocab ln:       {expected_initial:.2f} (esperado loss inicial)')
    print(f'  tempo total:    {total_min:.1f}min')
    if step_times:
        print(f'  step médio:     {sum(step_times)/len(step_times):.2f}s')

## 8. Geração DEPOIS — comparação

Se passou no verdict, as continuações devem mencionar **padrões dos templates sintéticos** (CNPJ, CPF, R$, datas, exercício 2024, etc.), não texto aleatório.

In [ ]:
for p in BASELINE_PROMPTS:
    print(f'\n{"="*70}')
    print(f'PROMPT: {p!r}')
    print(f'\n[ANTES] {baseline[p][:200]}')
    try:
        after = engine.generate(p, MAX_TOKENS, TEMP)
    except Exception as e:
        after = f'(erro: {type(e).__name__}: {e})'
    print(f'\n[DEPOIS] {after[:200]}')

## 9. Próximos passos

**Se passou (✅):** stack confirmado funcional. Próximos:
- Rodar pre-training real com dados maiores (notebook `oxta_contabil_pretrain.ipynb`)
- Pegar HF_TOKEN pra incluir CulturaX (300B+ tokens de PT-BR)
- Escalar pra 1B params (`d_model=1024, num_layers=16`)

**Se warn (⚠️):** stack tá vivo mas aprendeu pouco. Tentar:
- `NSOS_SMOKE_STEPS=1000` (mais steps)
- Dataset sintético maior (`N_SENTENCES=5000`)

**Se falhou (❌):** algum kernel quebrou. Diagnóstico:
- NaN/Inf → desligar BF16 (`NSOS_MIXED_PRECISION=fp32`)
- Crash CUDA → reduzir `d_model=512`
- `load_model` False → algum campo do ModelConfig inválido pra esse binding